In [ ]:
!pip install gensim numpy gdown

In [ ]:
# Download
!wget -c https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
!wget -c https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ru.300.vec.gz


--2025-08-17 06:50:38--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.171.22.33, 3.171.22.118, 3.171.22.68, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.171.22.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1325960915 (1.2G) [binary/octet-stream]
Saving to: ‘cc.en.300.vec.gz’

cc.en.300.vec.gz    100%[===================>]   1.23G  74.4MB/s    in 17s     

2025-08-17 06:50:56 (75.4 MB/s) - ‘cc.en.300.vec.gz’ saved [1325960915/1325960915]



In [5]:
from gensim.models import KeyedVectors

en_emb = KeyedVectors.load_word2vec_format("cc.en.300.vec.gz", limit=100000)
ru_emb = KeyedVectors.load_word2vec_format("cc.ru.300.vec.gz", limit=100000)

In [6]:
ru_emb.most_similar([ru_emb["август"]], topn=10)
en_emb.most_similar([en_emb["august"]], topn=10)

[('august', 0.9999999403953552),
 ('september', 0.8252838850021362),
 ('october', 0.8111193180084229),
 ('june', 0.8050147891044617),
 ('july', 0.797055184841156),
 ('november', 0.788363516330719),
 ('february', 0.7831973433494568),
 ('december', 0.7824540138244629),
 ('january', 0.7743154168128967),
 ('april', 0.7621643543243408)]

In [25]:
!wget -O en-ru.txt https://dl.fbaipublicfiles.com/arrival/dictionaries/en-ru.txt

# Rename it to match your notebook's expected file
!mv en-ru.txt /content/en_rus.train.txt

import random

# Read the downloaded dictionary
with open("/content/en_rus.train.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

random.shuffle(lines)
split_idx = int(len(lines) * 0.95)

# Train
with open("/content/en_rus.train.txt", "w", encoding="utf-8") as f:
    f.writelines(lines[:split_idx])

# Test
with open("/content/en_rus.test.txt", "w", encoding="utf-8") as f:
    f.writelines(lines[int(len(lines) * 0.95):])

--2025-08-17 08:18:15--  https://dl.fbaipublicfiles.com/arrival/dictionaries/en-ru.txt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.167.112.53, 3.167.112.129, 3.167.112.51, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.167.112.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1231068 (1.2M) [text/plain]
Saving to: ‘en-ru.txt’

en-ru.txt           100%[===================>]   1.17M  6.76MB/s    in 0.2s    

2025-08-17 08:18:15 (6.76 MB/s) - ‘en-ru.txt’ saved [1231068/1231068]



In [26]:
import numpy as np

def load_word_pairs(filename):
    en_ru_pairs = []
    en_vectors = []
    ru_vectors = []
    with open(filename, "r", encoding="utf-8") as inpf:
        for line in inpf:
            en, ru = line.rstrip().split(" ")
            if en not in en_emb or ru not in ru_emb:
                continue
            en_ru_pairs.append((en, ru))
            en_vectors.append(en_emb[en])
            ru_vectors.append(ru_emb[ru])
    return en_ru_pairs, np.array(en_vectors), np.array(ru_vectors)

en_ru_train, X_train, Y_train = load_word_pairs("en_rus.train.txt")
en_ru_test, X_test, Y_test = load_word_pairs("en_rus.test.txt")


In [27]:
# OLS mapping

from sklearn.linear_model import LinearRegression

mapping = LinearRegression(fit_intercept=False)
mapping.fit(X_train, Y_train)

# "august" -> nearest russian word (names of the month expected)
august = mapping.predict(en_emb["august"].reshape(1, -1))
hello = mapping.predict(en_emb["Hello"].reshape(1, -1))
ru_emb.most_similar([august[0]])
ru_emb.most_similar([hello[0]])

def translate_ols(sentence, topn=1):
    words = sentence.strip().split()
    translated = []

    for word in words:
        if word not in en_emb:
            translated.append(word)
            continue
        mapped_vec = mapping.predict(en_emb[word].reshape(1, -1))[0]
        nearest_ru = ru_emb.similar_by_vector(mapped_vec, topn=topn)[0][0]
        translated.append(nearest_ru)

    return " ".join(translated)
print(translate_ols('.'))
print(translate_ols("1 , 3"))

print(translate_ols("hello my name is Firdavs, what is your name")) # simple ols translation

....
1 , 3
здравствуй мой имя это Firdavs, что это ваш имя


In [28]:
# percision function
from scipy.spatial.distance import cdist

def precision(pairs, mapped_vectors, topn=1):
    assert len(pairs) == len(mapped_vectors)
    num_matches = 0
    all_ru_words = list(ru_emb.index_to_key)
    all_ru_vectors = ru_emb.vectors
    for i, (_, ru_word) in enumerate(pairs):
        distances = cdist([mapped_vectors[i]], all_ru_vectors, metric='cosine')[0]
        nearest_idx = np.argsort(distances)[:topn]
        nearest_words = [all_ru_words[j] for j in nearest_idx]
        if ru_word in nearest_words:
            num_matches += 1
    return num_matches / len(pairs)


In [2]:
#assert precision([("august", "август")], august, topn=5) == 0.0
#assert precision([("august", "август")], august, topn=9) == 1.0
#assert precision([("august", "август")], august, topn=10) == 1.0

precision_top1 = precision(en_ru_test, mapping.predict(X_test), 1)
precision_top5 = precision(en_ru_test, mapping.predict(X_test), 5)
print(precision_top1, precision_top5)

NameError: name 'en_ru_test' is not defined

In [31]:
# Orthogonal Procrustes solution (SVD)
import numpy as np
from scipy.spatial.distance import cdist

# Normalizing
X_train_norm = X_train / np.linalg.norm(X_train, axis=1, keepdims=True)
Y_train_norm = Y_train / np.linalg.norm(Y_train, axis=1, keepdims=True)
X_test_norm = X_test / np.linalg.norm(X_test, axis=1, keepdims=True)

def learn_transform(X, Y):
    # X W ≈ Y
    U, _, Vt = np.linalg.svd(X.T @ Y)
    W = U @ Vt
    return W

W = learn_transform(X_train_norm, Y_train_norm)

mapped_X_test = X_test_norm @ W

precision_top1_ortho = precision(en_ru_test, mapped_X_test, topn=1)
precision_top5_ortho = precision(en_ru_test, mapped_X_test, topn=5)

print(f"Top-1 precision: {precision_top1_ortho:.4f}")
print(f"Top-5 precision: {precision_top5_ortho:.4f}")

Top-1 precision: 0.3708
Top-5 precision: 0.6985


In [32]:
# FINALLY SIMPLE TRANSLATOR

def translate(sentence):
    words = sentence.strip().split()
    translated = []
    for word in words:
        if word not in en_emb:
            translated.append(word)
            continue
        vec = en_emb[word] / np.linalg.norm(en_emb[word])  # normalize
        mapped_vec = vec @ W
        nearest_ru = ru_emb.similar_by_vector(mapped_vec, topn=1)[0][0]
        translated.append(nearest_ru)
    return " ".join(translated)

# Examples
print(translate("."))
print(translate("1 , 3"))
print(translate("hello my name is Firdavs"))

.
1 , 6
привет мой имя является Firdavs
